# Lab 6
- **Student Name**: Daniel Alejandro Castillo Martin
- **Student ID**: N10019284
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
---

## Overview

This lab investigates parameter-efficient fine-tuning (PEFT) by adapting the pretrained **HuggingFaceTB/SmolLM2-135M** language model to a text classification task using Low-Rank Adaptation (LoRA).

The experiments compare full fine-tuning against LoRA at different ranks, evaluating classification accuracy, trainable parameter counts, computational cost, and memory consumption.

## Objectives

1. Evaluate the pre-trained model using a zero-shot classification baseline.
2. Implement answer-only loss masking to focus training on classification labels.
3. Implement LoRA from scratch and validate its behavior against the PEFT library.
4. Compare full fine-tuning with LoRA ranks 1, 4, and 16 using controlled experiments.
5. Predict and measure memory consumption across fine-tuning configurations.
6. Analyze the trade-offs between accuracy, parameter efficiency, and computational resources.


## 1. Imports and Setup




- This section initializes the libraries, configuration parameters, and execution environment required for the experiments.

- The implementation uses PyTorch for tensor operations and model training, Hugging Face Transformers for pretrained language models, and PEFT for validating the custom LoRA implementation.

- The provided `lab06_starter.py` module supplies standardized dataset loading, training, evaluation, and memory-measurement utilities, ensuring consistent experimental conditions.

### 1.1 Required Libraries

In [1]:
# Import general libraries for reproducibility and paths
import sys
import subprocess

# Import libraries for data analysis and visualization
import matplotlib.pyplot as plt
import pandas as pd

# Import PyTorch for model operations and verification
import torch
import torch.nn as nn
import torch.nn.functional as F

# Import the provided lab starter
from scripts import lab06_starter as S

# Import experiment configuration
from src.config import (MODEL_NAME, TASK_NAME, SEEDS, DATA_SEED, N_TRAIN, N_TEST, LORA_RANKS, TRAINING_STEPS, BATCH_SIZE, FULL_LR, LORA_LR, OUTPUT_DIR, FIGURES_DIR, create_output_directories)

# Import reproducibility and hardware utilities
from src.utils import set_seed, get_hardware_info


# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

C:\Users\dancm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Experiment Configuration


- The experiment settings are defined centrally to ensure consistency and reproducibility across fine-tuning methods.

- All configurations will use the same dataset, evaluation split, training steps, batch size, and random seeds. Learning rates may differ between full fine-tuning and LoRA, as permitted by the assignment.

- Generated results and figures will be stored separately from the source code.

In [2]:
# Initialize the experiment output directories.
create_output_directories()

# Display the experimental configuration.
print(f'Model: {MODEL_NAME}')
print(f'Task: {TASK_NAME}')
print(f'Training examples: {N_TRAIN}')
print(f'Test examples: {N_TEST}')
print(f'Data seed: {DATA_SEED}')
print(f'Training seeds: {SEEDS}')
print(f'LoRA ranks: {LORA_RANKS}')
print(f'Training steps: {TRAINING_STEPS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Full fine-tuning LR: {FULL_LR}')
print(f'LoRA LR: {LORA_LR}')

Model: HuggingFaceTB/SmolLM2-135M
Task: emotion
Training examples: 600
Test examples: 500
Data seed: 0
Training seeds: [0, 1, 2]
LoRA ranks: [1, 4, 16]
Training steps: 120
Batch size: 8
Full fine-tuning LR: 5e-05
LoRA LR: 0.001


### 1.3 Reproducibility and Hardware

- Reproducibility is important for ensuring that observed differences between full fine-tuning and LoRA configurations are attributable to the methods being compared rather than uncontrolled randomness.

- The experiments use fixed random seeds for Python and PyTorch, consistent dataset sampling, and identical training budgets across configurations. Each experiment will be repeated using seeds 0, 1, and 2.

- The available hardware and software environment are also recorded because training time and GPU memory consumption depend on the execution platform.

In [3]:
# Initialize reproducibility with the first experimental seed
set_seed(SEEDS[0])

# Retrieve and display the execution environment
hardware_info = get_hardware_info()

for key, value in hardware_info.items():
    print(f'{key}: {value}')

Python: 3.12.10
PyTorch: 2.8.0+cu128
CUDA available: True
CUDA version: 12.8
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


### 1.4 Starter Script Validation

- Before implementing the custom LoRA components, the provided `lab06_starter.py` script is executed without modification to validate the experimental environment.

- The starter loads the pretrained SmolLM2-135M model and the emotion classification dataset, evaluates zero-shot accuracy, and performs a short 30-step LoRA training run using PEFT.

- This preliminary test verifies that model loading, dataset preparation, evaluation, and GPU-accelerated training work correctly before conducting the controlled experiments.

In [4]:
# Configure execution to test that the starter script works
result = subprocess.run([sys.executable, 'scripts/lab06_starter.py'], capture_output = True, text = True, encoding = 'utf-8',errors = 'replace')

# Display the execution output
print(result.stdout)

# Validate successful execution
assert result.returncode == 0, 'Starter script execution failed.'

print('Starter script validation completed successfully.')

device: cuda
example prompt: 'Text: i may feel that way but the fact that stories created by adults that are meant for children contain messages that are not so innocent really makes me wonder who exactly is\nEmotion:' answer: ' joy'
zero-shot accuracy (chance 0.167): 0.35
LoRA r=8, 30 steps: accuracy 0.510, trainable 460,800 of 134,975,808, 0.12 s/step

Starter script validation completed successfully.


#### Validation Results

The original starter script completed successfully using CUDA acceleration on the NVIDIA GeForce RTX 3060 Laptop GPU.

- **Zero-shot accuracy:** 35.0%, exceeding the random-chance baseline of 16.7%.
- **LoRA accuracy:** 51.0% after 30 training steps, representing a 16-percentage-point improvement.
- **Trainable parameters:** 460,800 out of 134,975,808 total parameters (approximately 0.34%).
- **Training time:** Approximately 0.12 seconds per step.

The results confirm that pretrained model loading, dataset preparation, evaluation, and LoRA fine-tuning are operational.

Although the preliminary LoRA run improved classification accuracy, this result alone does not establish whether LoRA matches full fine-tuning or how adaptation rank affects performance. Those comparisons require controlled experiments using identical datasets, training budgets, and random seeds.